# L2-Thinker vocabulary trim: verify every number from the shipped rows (no GPU)

This notebook recomputes the results in `docs/writeup.md` from the generation rows in `results/rows/`. It needs only pandas and `vendor/l2.py` (the MGSM answer parser). Runs in under a minute on a laptop or a free Colab CPU runtime.

What it reproduces: paired greedy identity (trimmed vs same-session control), truncation and loop-score deltas, MGSM accuracy with an exact McNemar test, the pre-registered checks, the E0 noise floor (control vs control), and the exploratory Swahili MGSM result. The L2 (wrong-language) rate needs FastText and the 125 MB `lid.176.bin`; it is optional below.

In [ ]:
import sys, subprocess, os
from pathlib import Path
# If running on Colab, clone the project repo; locally, run from the repo root.
if Path("../scripts/readout.py").exists(): os.chdir("..")          # launched from notebooks/
if not Path("scripts/readout.py").exists():
    REPO_URL = "https://github.com/ayastaga/l2thinker-trim"      # <- change if the repo lives elsewhere
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/l2thinker-trim"], check=True); os.chdir("/content/l2thinker-trim")
sys.path.insert(0, "scripts"); sys.path.insert(0, "vendor")
import readout
print("repo:", os.getcwd())

In [ ]:
# Every comparison, with pre-registered checks. Writes results/tables/*.csv.
import pandas as pd; pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
for tag, (label, in_region) in readout.COMPARISONS.items():
    p, gpu = readout.pair(tag); res = readout.comparison_table(p, in_region)
    print(f"\n== {label}\n   gpu={gpu} paired={len(p)} identical={int(p.same.sum())} ({100*p.same.mean():.2f}%)")
    display(res.round(3))
    for name, vals, ok in readout.checks(res, bool(in_region)): print(f"   {name}: {vals} -> {'PASS' if ok else 'FAIL'}")

In [ ]:
# E0: the control regenerated in a second session on a different A100 instance. 900/900 identical => zero noise floor.
t, ident, n = readout.noise_floor(); display(t.round(1)); print(f"control-vs-control identity: {ident:.2f}% over {n} rows")

In [ ]:
# The eight in-region divergences, side by side at the first differing character.
for tag in ("trim_a100", "trim_a100_r2"):
    p, _ = readout.pair(tag)
    for _, r in p[~p.same].iterrows():
        if r.lang == "sw": continue
        f = "trace" if r.trace_c != r.trace_t else "answer"; a, b = r[f"{f}_c"] or "", r[f"{f}_t"] or ""
        k = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
        print(f"\n{tag} {r.id} {r.precision} field={f} char={k}\n  ctrl: {a[max(0,k-50):k+50]!r}\n  trim: {b[max(0,k-50):k+50]!r}")

In [ ]:
# Swahili MGSM on the South Asia trim: the out-of-region cost, by the numbers.
p, _ = readout.pair("trim_a100_r2x")
a, b = p.correct_c.astype(bool).values, p.correct_t.astype(bool).values
print(f"accuracy ctrl {100*a.mean():.1f}% -> trim {100*b.mean():.1f}% | truncated {100*p.truncated_c.mean():.1f}% -> {100*p.truncated_t.mean():.1f}% | "
      f"loop {p.doomloop_c.mean():.3f} -> {p.doomloop_t.mean():.3f} | McNemar p = {readout.mcnemar_p(a, b):.2e} | discordant {int((a != b).sum())}")

In [ ]:
# Optional: L2 rate (predominant language of the trace == prompt language). Needs fasttext + lid.176.bin (~125 MB).
# !pip -q install fasttext-wheel && wget -q -O /content/lid.176.bin https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
# import l2; ct = readout.rows_df(readout.CTRL, "trim_a100"); tr = readout.rows_df(readout.TRIM, "trim_a100")
# for d in (ct, tr): d["trace_lang"] = [l2.lid(t, root="/content") if isinstance(t, str) and t.strip() else None for t in d.trace]; d["l2"] = d.trace_lang == d.lang
# print(ct.groupby(["dataset","precision","lang"]).l2.mean().mul(100).round(1)); print(tr.groupby(["dataset","precision","lang"]).l2.mean().mul(100).round(1))